# Science Hallucination Detector
vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference. As a demonstration of that, in this notebook, we show how vLLM-Hook enables *scientific-claim hallucination detector*.

**Paper**: [Detecting Hallucinations in Scientific Claims by Combining Prompting Strategies and Internal State Classification](https://aclanthology.org/2025.sdp-1.30/).<br />
**Authors**: Yupeng Cao, Chun-Nam Yu, K.P. Subbalakshmi.<br />
**"TL;DR"**: Each (claim, reference) pair is fed to an LLM with a few-shot prompt asking for a justification and classification (*entailment*, *contradiction*, or *unverifiable*). The model's hidden state at the last response token is then classified by a small logistic-regression model trained in advance.

### Installation
If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [ ]:
from pathlib import Path
import sys

# vllm_hooks/notebooks/
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT/"vllm_hook_plugins"
REQ_FILE = REPO_ROOT/"requirement.txt"

print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

%pip install -e "{PKG_DIR}"

if REQ_FILE.exists():
    %pip install -r "{REQ_FILE}"
else:
    print("\u26a0\ufe0f requirements.txt not found at", REQ_FILE)

### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like `vllm.LLM` but adds support for hooks and instrumentation. We also import `TokensPrompt` because this demo feeds the engine pre-tokenized inputs (see the prompt-builder section).

In [1]:
from vllm import SamplingParams, TokensPrompt
from vllm_hook_plugins import HookLLM

/proj/dmfexp/irene/miniconda3/envs/vllm_hook_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Environment & multiprocessing setup

In [2]:
import json
import os
import multiprocessing as mp
import torch
mp.set_start_method("spawn", force=True)
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

### Helpers: dataset fetcher and prompt builder
The SciHal-2025 train and test splits are committed to [InfintyLab/SciHal-Challenge](https://github.com/InfintyLab/SciHal-Challenge) on GitHub. We fetch each split once and cache it locally.

In [3]:
SciHal_url = (
    "https://raw.githubusercontent.com/InfintyLab/SciHal-Challenge/"
    "main/data/dataset/"
)

def load_scihal_split(cache_dir: str, filename: str) -> list:
    """Download SciHal-Challenge dataset if no cache in local."""
    local = os.path.join(cache_dir, filename)
    if not os.path.exists(local):
        from urllib.request import urlretrieve
        print(f"Downloading SciHal to {local}")
        urlretrieve(SciHal_url + filename, local)
    with open(local) as f:
        return json.load(f)

Following the original SciHal setup, we build a few-shot prefix from the train split with 2 examples per label.

In [4]:
def build_few_shot_middle(train_dataset: list, count_target: int = 2, total_target: int = 6) -> str:
    """Build few shot examples following https://github.com/InfintyLab/SciHal-Challenge/blob/97817c788bde330c61f8ed2fc750d83b19fa1590/llama3_finetunec3_just_task1.py#L141."""
    middle = ""
    count_dict = {"entailment": 0, "contradiction": 0, "unverifiable": 0}
    total = 0
    for x in train_dataset:
        label = x["label"]
        if count_dict.get(label, 0) >= count_target:
            continue
        my_input = "#Claim: " + x["claim"] + "\n #Reference: " + x["reference"]
        middle += (
            "Input: " + my_input + "\n\n"
            "Output:\n" + x["justification"] + "\n#Label: " + label + "\n\n"
        )
        count_dict[label] += 1
        total += 1
        if total == total_target:
            break
    return middle

Build SciHal prompt token IDs following the original authors' doubled-BOS practice: apply_chat_template returns a string already prefixed with <|begin_of_text|>, then tokenizer() with add_special_tokens=True prepends ANOTHER <|begin_of_text|>. The classifier was trained on hidden states extracted from this prefix.

In [5]:
PROMPT_TEMPLATE_PREFIX = (
    "\n"
    "You are a helpful assistant. Learn from the examples below and complete the task accordingly. \n"
    "\n"
    "### Task: Detect if the claims are well-supported by the references. Provide a justification and classify each example into three labels: entailment, contradiction, or unverifiable\n"
    "\n"
)

PROMPT_TEMPLATE_SUFFIX = (
    "\n"
    "\n"
    "### Now, apply the same pattern: \n"
    "\n"
    "Input: !INPUT!\n"
    "Output: \n"
)

def build_prompt_ids(tokenizer, few_shot_middle: str, claim: str, reference: str) -> list:
    my_input = "#Claim: " + claim + "\n #Reference: " + reference
    user_msg = few_shot_middle + PROMPT_TEMPLATE_SUFFIX.replace("!INPUT!", my_input)
    chat = [
        {"role": "system", "content": PROMPT_TEMPLATE_PREFIX},
        {"role": "user", "content": user_msg},
    ]
    message = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    return tokenizer(message, add_special_tokens=True).input_ids

### Initialize `HookLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [6]:
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
model = "meta-llama/Llama-3.1-8B-Instruct"

n_test = 9  # number of test items to classify

dtype_map = {
    'meta-llama/Llama-3.1-8B-Instruct': torch.float16,
}

We also need to provide a config file that specifies which decoder layer to capture and which classifier to use. For science hallucination detection, this config can be obtained from https://github.com/InfintyLab/SciHal-Challenge/blob/main/llama3_finetunec3_just_task1.py.

In [7]:
from pathlib import Path

json_path = Path(f"../model_configs/hidden_states/{model.split('/')[-1]}.json")

with open(json_path) as f:
    config = json.load(f)
clf_path = config["scihal"]["clf_path"]

Inside `probe_hidden_states` and `science_hallucination` we define the desired behavior during model inference and after the model inference:
- `workers/probe_hidden_states_worker.py` defines that we need pre-final-norm residual at the requested token mode
- `analyzers/science_hallucination_analyzer.py` runs the saved classifier on the model's final RMSNorm.

Now, we initialize the LLM:

In [8]:
llm = HookLLM(
    model=model,
    worker_name="probe_hidden_states",
    analyzer_name="science_hallucination",
    config_file=json_path,
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_prefix_caching=True,
    enable_hook=True
)

INFO 06-07 17:59:01 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': '/u/cyko1/.cache/huggingface/hub', 'dtype': torch.float16, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'worker_extension_cls': 'vllm_hook_plugins.workers.probe_hidden_states_worker.ProbeHiddenStatesWorker', 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
WARNING 06-07 17:59:01 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 06-07 17:59:01 [model.py:549] Resolved architecture: LlamaForCausalLM
WARNING 06-07 17:59:01 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 06-07 17:59:01 [model.py:1678] Using max model len 131072


2026-06-07 17:59:02,092	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 06-07 17:59:02 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 06-07 17:59:02 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 06-07 17:59:02 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-07 17:59:02 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-07 17:59:02 [vllm.py:1025] Cudagraph is disabled under eager mode
INFO 06-07 17:59:02 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant
(EngineCore pid=2725770) INFO 06-07 17:59:14 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_c

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:10<00:31, 10.43s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:23<00:24, 12.16s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:36<00:12, 12.36s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:37<00:00,  7.73s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:37<00:00,  9.26s/it]
(EngineCore pid=2725770) 


(EngineCore pid=2725770) INFO 06-07 17:59:54 [default_loader.py:384] Loading weights took 37.05 seconds
(EngineCore pid=2725770) INFO 06-07 17:59:55 [gpu_model_runner.py:4820] Model loading took 14.99 GiB memory and 38.203916 seconds
(EngineCore pid=2725770) INFO 06-07 17:59:57 [gpu_worker.py:436] Available KV cache memory: 38.26 GiB
(EngineCore pid=2725770) INFO 06-07 17:59:57 [kv_cache_utils.py:1319] GPU KV cache size: 313,408 tokens
(EngineCore pid=2725770) INFO 06-07 17:59:57 [kv_cache_utils.py:1324] Maximum concurrency for 131,072 tokens per request: 2.39x


(EngineCore pid=2725770) 2026-06-07 17:59:57,408 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2725770) 2026-06-07 17:59:57,430 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=2725770) INFO 06-07 17:59:57 [core.py:283] init engine (profile, create kv cache, warmup model) took 2.38 seconds
(EngineCore pid=2725770) INFO 06-07 17:59:59 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=2725770) WARNING 06-07 17:59:59 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=2725770) WARNING 06-07 17:59:59 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=2725770) INFO 06-07 17:59:59 [vllm.py:1025] Cudagraph is disabled under eager mode
(EngineCore pid=2725770) INFO 06-07 17:59:59 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant


### n_test Test cases
We use the first `n_test` items from `subtask1_test.json` and build the few-shot prefix from `subtask1_train_batch3.json`.

In [9]:
tokenizer = llm.tokenizer
train = load_scihal_split(cache_dir, "subtask1_train_batch3.json")
few_shot_middle = build_few_shot_middle(train)
test_cases = load_scihal_split(cache_dir, "subtask1_test.json")[:n_test]
prompt_ids_list = [
    build_prompt_ids(tokenizer, few_shot_middle, q["claim"], q["reference"]) for q in test_cases
]
print(f"prepared {len(prompt_ids_list)} prompts; example length = {len(prompt_ids_list[0])} tokens")

prepared 9 prompts; example length = 3398 tokens


We run a normal generation pass with hooks disabled to generate the response.

In [10]:
gen_outputs = llm.generate(
    [TokensPrompt(prompt_token_ids=ids) for ids in prompt_ids_list],
    SamplingParams(temperature=0.0, max_tokens=1024),
    use_hook=False,
)
response_token_ids = [list(gen_output.outputs[0].token_ids) for gen_output in gen_outputs]

Rendering prompts: 100%|████████████████████████████████████████████████| 9/9 [00:00<00:00, 87.60it/s]
Processed prompts: 100%|█| 9/9 [00:01<00:00,  6.56it/s, est. speed input: 23090.21 toks/s, output: 620


We then re-prefill with `prompt + response[:-2]` and capture the last-token hidden state following https://github.com/InfintyLab/SciHal-Challenge/blob/97817c788bde330c61f8ed2fc750d83b19fa1590/llama3_finetunec3_just_task1.py#L262.

In [11]:
capture_prompts = [
    TokensPrompt(prompt_token_ids=list(p) + list(r[:-2]))
    for p, r in zip(prompt_ids_list, response_token_ids)
]
output = llm.generate(capture_prompts, SamplingParams(temperature=0.0, max_tokens=1), save_to_disk=True)

Rendering prompts: 100%|██████████████████████████████████████████████| 9/9 [00:00<00:00, 3910.17it/s]
Processed prompts: 100%|█| 9/9 [00:00<00:00, 150.22it/s, est. speed input: 549754.40 toks/s, output: 1


Finally we classify with the science_hallucination analyzer.

In [12]:
LABEL_NAMES = ["entailment", "contradiction", "unverifiable"]
spec = {
    "label_names": LABEL_NAMES,
    "clf_path": clf_path,
    "model_id": model,
}
stats = llm.analyze(probes=getattr(output[0], "probes", None), analyzer_spec=spec)

labels = stats["prediction_labels"]
print("=" * 50)
for case, label in zip(test_cases, labels):
    print(f"classifier label: {label}")

/proj/dmfexp/irene/miniconda3/envs/vllm_hook_env/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Fetching 6 files: 100%|██████████████████████████████████████████████| 6/6 [00:00<00:00, 63072.24it/s]


classifier label: contradiction
classifier label: contradiction
classifier label: entailment
classifier label: contradiction
classifier label: unverifiable
classifier label: entailment
classifier label: entailment
classifier label: contradiction
classifier label: entailment


### (Optional) User can also try out rpc path which allows easier debugging

In [13]:
output = llm.generate(capture_prompts, SamplingParams(temperature=0.0, max_tokens=1), save_to_disk=False)
stats = llm.analyze(probes=getattr(output[0], "probes", None), analyzer_spec=spec)

labels = stats["prediction_labels"]
print("=" * 50)
for case, label in zip(test_cases, labels):
    print(f"classifier label: {label}")

Rendering prompts: 100%|██████████████████████████████████████████████| 9/9 [00:00<00:00, 4005.60it/s]
Processed prompts: 100%|█| 9/9 [00:00<00:00, 168.92it/s, est. speed input: 617678.58 toks/s, output: 1


classifier label: contradiction
classifier label: contradiction
classifier label: entailment
classifier label: contradiction
classifier label: unverifiable
classifier label: entailment
classifier label: entailment
classifier label: contradiction
classifier label: entailment
